# 00 — IFC Inventory

Sanity check that both source IFC models open in `ifcopenshell` and contain instances of every entity class targeted by `ids/mep_services_v1.ids` before the validator is run against them.

**Models under test**

| File | Source | Expected schema |
|---|---|---|
| `ifc/Ifc4_Revit_MEP.ifc` | youshengCode/IfcSampleFiles — Autodesk Revit MEP Advanced Sample export | IFC4 |
| `ifc/BoilerGasRadiatorDomesticHotWater.ifc` | EnEff-BIM/EnEffBIM_UseCases — VDI 6020 reference (MIT) | IFC4 |

The notebook can be re-run any time the IFCs are refreshed.

In [1]:
from __future__ import annotations

from pathlib import Path
from collections import OrderedDict

import ifcopenshell
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IFC_DIR = REPO / "ifc"

MODELS = OrderedDict([
    ("Primary  (Revit MEP)", IFC_DIR / "Ifc4_Revit_MEP.ifc"),
    ("Secondary (Boiler/DHW)", IFC_DIR / "BoilerGasRadiatorDomesticHotWater.ifc"),
])

# Entity classes referenced by the 9 MEP IDS specifications.
IDS_ENTITIES = [
    "IfcPipeSegment",
    "IfcDistributionPort",
    "IfcFlowTerminal",
    "IfcFireSuppressionTerminal",
    "IfcPump",
    "IfcDuctSegment",
    "IfcValve",
    "IfcCableSegment",
    "IfcElectricAppliance",
]

for label, path in MODELS.items():
    assert path.exists(), f"Missing IFC file: {path}"

print("All IFC files located.")

All IFC files located.


## Open models and report schema + size

In [2]:
opened = {}
summary_rows = []
for label, path in MODELS.items():
    model = ifcopenshell.open(str(path))
    opened[label] = model
    summary_rows.append({
        "Model": label,
        "Path": path.name,
        "Schema": model.schema,
        "Size (MB)": round(path.stat().st_size / (1024 * 1024), 2),
        "Total entities": len(list(model)),
    })

pd.DataFrame(summary_rows).set_index("Model")

,Path,Schema,Size (MB),Total entities
Model,,,,
Primary (Revit MEP),Ifc4_Revit_MEP.ifc,IFC4,27.81,302453
Secondary (Boiler/DHW),BoilerGasRadiatorDomesticHotWater.ifc,IFC4,1.35,26030


## Entity counts for the 9 MEP IDS specifications

In [3]:
rows = []
for label, model in opened.items():
    for entity in IDS_ENTITIES:
        try:
            count = len(model.by_type(entity))
        except Exception:
            count = 0
        rows.append({"Model": label, "Entity": entity, "Count": count})

counts_df = pd.DataFrame(rows).pivot(index="Entity", columns="Model", values="Count").fillna(0).astype(int)
counts_df.loc["-- TOTAL --"] = counts_df.sum()
counts_df

Model,Primary (Revit MEP),Secondary (Boiler/DHW)
Entity,,
IfcCableSegment,0,0
IfcDistributionPort,8515,114
IfcDuctSegment,837,0
IfcElectricAppliance,0,0
IfcFireSuppressionTerminal,6,0
IfcFlowTerminal,736,2
IfcPipeSegment,491,29
IfcPump,0,1
IfcValve,0,4


### Coverage interpretation

Specifications whose entity class has zero matches in both models will report
`0 applicable entities` and contribute neither a pass nor a fail. This is
expected behaviour for IDS and is discussed in the technical report.

## IfcFireSuppressionTerminal PredefinedType distribution

Spec MEP-04 expects PredefinedType to be one of
`{BREECHINGINLET, FIREHYDRANT, HOSEREEL, SPRINKLER, SPRINKLERDEFLECTOR, USERDEFINED}`.
Anything `NOTDEFINED` will fail validation.

In [4]:
for label, model in opened.items():
    try:
        terminals = model.by_type("IfcFireSuppressionTerminal")
    except Exception:
        terminals = []
    if not terminals:
        print(f"{label}: no IfcFireSuppressionTerminal instances.")
        continue
    dist = {}
    for t in terminals:
        pt = getattr(t, "PredefinedType", None) or "<null>"
        dist[pt] = dist.get(pt, 0) + 1
    print(f"{label}: {dist}")

Primary  (Revit MEP): {'NOTDEFINED': 6}
Secondary (Boiler/DHW): no IfcFireSuppressionTerminal instances.


## Authoring tool fingerprint

Useful in the report to show which exporter produced each IFC.

In [5]:
for label, model in opened.items():
    h = model.header
    org = h.file_name.organization if h.file_name and h.file_name.organization else "<unknown>"
    app = h.file_name.preprocessor_version if h.file_name and h.file_name.preprocessor_version else "<unknown>"
    schema = h.file_schema.schema_identifiers if h.file_schema else ["<unknown>"]
    print(f"{label}")
    print(f"  organisation: {org}")
    print(f"  preprocessor: {app}")
    print(f"  schema:       {schema}")

Primary  (Revit MEP)
  organisation: ('',)
  preprocessor: The EXPRESS Data Manager Version 5.02.0100.07 : 28 Aug 2013
  schema:       ('IFC4',)
Secondary (Boiler/DHW)
  organisation: ('',)
  preprocessor: The EXPRESS Data Manager Version 5.02.0100.07 : 28 Aug 2013
  schema:       ('IFC4',)
